# Synthetic Identity Fraud Detection

### Fraud Detection Aml Analytics — Banking-AI

This notebook explains each step in plain language so new learners can follow:

## Step 0 — Notebook Header

**Prerequisites:** Basic Python (`pandas`, `numpy`, `matplotlib`), familiarity with `scikit-learn` basics, and basic understanding of fraud detection and anti-money laundering terminology.

**What You Will Learn:**

After completing this notebook, you will be able to:
- Explain the fraud detection and anti-money laundering problem and why the chosen classification approach suits this banking workflow.
- Implement an end-to-end classification pipeline on synthetic data and evaluate performance with domain-appropriate metrics.
- Assess whether the model is operationally viable, considering error costs, fairness, and the limitations of synthetic data.

**Estimated time:** 35–45 minutes

**Collection context:** Fraud detection, AML compliance, anomaly detection, and financial crime analytics in banking.

## Step 1 — Banking Problem Introduction

**Why this notebook matters:** Synthetic Identity Fraud is when a criminal combines real (often stolen) and fake information to create a new, "synthetic" person. This is harder to detect than traditional identity theft because there is no single victim to report the crime.

**Input data used:** SSN/ID consistency, email age, phone-to-address matching, and credit history (or lack thereof).

**What we predict:** Probability that an identity is synthetic (`is_synthetic`).

**ML method used:** Gradient Boosting (XGBoost/LightGBM style) - here we use `RandomForestClassifier` for broad compatibility.

**Learning goal:** Learn how to engineer "consistency features" to find sophisticated fraud.

**Primary stakeholders:** fraud analysts, compliance officers, risk managers, and financial crime investigators.

**Comprehension check:** Before looking at the data, write one sentence describing the core decision this model supports and who benefits from getting it right.

**Domain framing note:** This notebook uses synthetic data for educational purposes. It demonstrates decision-support prototyping, not production-ready deployment.

## Step 2 — Environment Setup

Import libraries and set deterministic seeds for reproducibility.

In [ ]:
# Requires: numpy>=1.24, pandas>=2.0, scikit-learn>=1.3, matplotlib>=3.7, seaborn>=0.13

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_curve, auc, accuracy_score

sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (10, 6)

print("Environment ready.")


## Step 3 — Data Creation & Context

Build Synthetic Application Dataset

We simulate 5,000 account applications. Synthetic identities often have "shallow" credit files and inconsistent PII.

In [ ]:
n = 5000
rng = RNG

# Basic Features
age_at_app = rng.integers(18, 80, n)
credit_score = rng.normal(650, 100, n).clip(300, 850)
email_domain_score = rng.choice([1, 0.5, 0.2], n, p=[0.8, 0.15, 0.05]) # 1=Standard, 0.2=Disposable
pii_match_score = rng.uniform(0.7, 1.0, n) # How well SSN matches Name/DOB in records
phone_address_match = rng.choice([1, 0], n, p=[0.9, 0.1])

# Synthetic Identity Injection
# Indicators: High age but low credit history, low PII match, disposable email
is_synthetic = np.zeros(n)
synthetic_indices = rng.choice(range(n), size=int(n * 0.06), replace=False)
is_synthetic[synthetic_indices] = 1

# Adjust features for synthetic accounts
pii_match_score[synthetic_indices] *= rng.uniform(0.3, 0.6, len(synthetic_indices))
credit_score[synthetic_indices] = rng.normal(450, 50, len(synthetic_indices)).clip(300, 600)
phone_address_match[synthetic_indices] = rng.choice([0, 1], len(synthetic_indices), p=[0.8, 0.2])

df = pd.DataFrame({
    'age_at_app': age_at_app,
    'credit_score': credit_score,
    'email_domain_score': email_domain_score,
    'pii_match_score': pii_match_score,
    'phone_address_match': phone_address_match,
    'is_synthetic': is_synthetic
})

print(f"Applications: {n}")
print(f"Synthetic Identities: {int(is_synthetic.sum())}")

## Step 4 — Exploratory Data Analysis

EDA

Notice how synthetic identities cluster in the low PII match and low credit score regions.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x='pii_match_score', y='credit_score', hue='is_synthetic', alpha=0.6, data=df)
plt.title('PII Consistency vs Credit Score')
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Scatter plot titled "PII Consistency vs Credit Score". The chart highlights spatial separation or clustering patterns in the data.

## Step 5 — Feature Engineering

Prepare features for modelling. Domain-meaningful transformations help the model learn patterns aligned with banking practice.

In [ ]:
X = df.drop('is_synthetic', axis=1)
y = df['is_synthetic']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y)

## Step 6 — Baseline Establishment

A baseline sets the floor that any useful model must beat. Without it, performance numbers have no context.

In [ ]:
# --- Baseline: always predict the majority class ---
baseline_clf = DummyClassifier(strategy='most_frequent', random_state=RANDOM_SEED)
baseline_clf.fit(X_train, y_train)
baseline_pred = baseline_clf.predict(X_test)
baseline_acc = accuracy_score(y_test, baseline_pred)
print(f"Baseline accuracy (majority-class): {baseline_acc:.3f}")
print(f"Any useful model must beat this {baseline_acc:.3f} floor.")

## Step 7 — Model Training & Evaluation

Train the model and compare performance against the baseline.

**Prediction prompt:** Before viewing the results, what performance do you expect and why?

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED)
model.fit(X_train, y_train)

y_proba = model.predict_proba(X_test)[:, 1]

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.3f}')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve: Synthetic Identity Detection')
plt.legend()
plt.tight_layout()
plt.show()
plt.close()

print("Classification Report at default threshold:")
from sklearn.metrics import classification_report
print(classification_report(y_test, model.predict(X_test)))

**Alt-text:** Line chart titled "ROC Curve: Synthetic Identity Detection" with False Positive Rate on the x-axis and True Positive Rate on the y-axis. The chart shows temporal trends and the alignment between predicted and observed values.

## Step 8 — Interpretability & Explainability

Interpret the model in domain terms. A banking professional should be able to assess whether the model's reasoning is credible.

In [ ]:
importances = pd.Series(model.feature_importances_, index=X.columns)
importances.sort_values().plot(kind='barh')
plt.title('Feature Importance for Synthetic Identity Detection')
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Bar chart titled "Feature Importance for Synthetic Identity Detection". The chart ranks features or categories by magnitude to highlight dominant factors.

## Summary
Synthetic identity detection relies heavily on **external data consistency**. When an applicant's self-reported data (PII) doesn't match historical credit header data or internal banking records, the risk of it being a manufactured identity increases significantly.

What features are most indicative of a synthetic identity?

## Step 9 — Operational Application

Demonstrate how the model output feeds into a real banking workflow decision.

In [ ]:
# --- Scenario: score representative cases from the test set ---
print("Operational demonstration — model decision support:\n")
proba = model.predict_proba(X_test)[:, 1]
low_idx  = int(np.argmin(proba))
high_idx = int(np.argmax(proba))
mid_idx  = int(np.argsort(proba)[len(proba) // 2])

for label, idx in [("Low-risk", low_idx), ("Medium-risk", mid_idx), ("High-risk", high_idx)]:
    row = X_test.iloc[idx]
    prob = proba[idx]
    print(f"{label} case  predicted probability: {prob:.1%}")
    print(f"  Features: {dict(row.round(2))}")
    print()

print("A decision-maker would combine these scores with business rules and domain judgement.")

## Step 10 — Summary & Reflection

**What you accomplished:** You built an end-to-end fraud detection and anti-money laundering workflow — from problem framing through data creation, baseline comparison, model training, evaluation, and operational demonstration.

**Limitations:**

- The dataset is synthetic and cannot capture the full complexity of real banking data.
- Model performance on synthetic data does not guarantee equivalent performance in production.
- This notebook is an educational prototype. Real deployment requires validated data, regulatory review, and ongoing monitoring.

**Ethical reflection:**

- False positives can freeze legitimate accounts, causing financial hardship and customer distrust.
- Fraud models risk encoding biases against specific demographic groups or geographic regions.
- Transaction monitoring must balance fraud prevention with customer privacy rights.

**Reflection questions:**

1. What additional data sources or features would you need before trusting this model in a live banking environment?
2. How would the relative cost of false positives versus false negatives change the metric you optimise for?
3. How could you adapt this workflow to a related problem in fraud detection and anti-money laundering?

## References

- Scikit-learn documentation: [https://scikit-learn.org/stable/](https://scikit-learn.org/stable/)
- Project quality baseline: `QUALITY_STANDARD.md`
- Domain context drawn from public banking and financial services literature.